# Instalando as Bibliotecas Necessárias



In [1]:
# Installing the numpy, netcdf4, boto3 and gdal libraries
!pip install -q cartopy boto3 gdal salem rasterio pyproj geopandas descartes ultraplot

# download do arquivo "utilities_goes16e19.py"
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/utilities_goes16e19.py

# download da paleta de cores para o canal do infravermelho
!wget -c https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/ir.cpt

# Monta drive
from google.colab import drive
drive.mount('/content/drive')

# Caminho do diretório
dir = '/content/drive/MyDrive/PYHTON/00_GITHUB/4_SATELITE/codigos_imagem_satelite_REFERENCIA'

# Diretório de Saída
import os
dir_output = f'{dir}/output/SAT_05'
os.makedirs(dir_output, exist_ok=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 70.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.6 MB/s eta 0:00:00
--2026-01-27 19:17:13--  https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/utilities_goes16e19.py
Resolving github.com (github.com)... 140.82.112.4
Connecting to github.com (github.com)|140.82.112.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/evmpython/Minicurso_UFCG_nov_2025/main/ut

# Paínel em projeção retangular em T-Realçada

In [2]:
%%time
#========================================================================================================================#
#                                          IMPORTAÇÃO DAS BIBLIOTECAS
#========================================================================================================================#
from utilities_goes16e19 import download_CMI, remap, loadCPT
from matplotlib import cm
import ultraplot as uplt
import pandas as pd
import os
from datetime import datetime
import xarray as xr
import cartopy.io.shapereader as shpreader
import cartopy.crs as ccrs

#========================================================================================================================#
#                                        CRIA DIRETÓRIO DE ENTRADA E SAÍDA
#========================================================================================================================#
input = "/content/input"; os.makedirs(input, exist_ok=True)
output = dir_output

#========================================================================================================================#
#                                        DEFINE OS LIMITES DA IMAGEM
#========================================================================================================================#
# canal
band = '13'

# área desejada da imagem
lonmin, lonmax, latmin, latmax = -48.0, -43.0, -23.5, -20.0

# coloca os limites da área numa lista
extent = [lonmin, latmin, lonmax, latmax]

# cria moldura da figura
fig, ax = uplt.subplots(ncols=3, nrows=4, axheight=6, tight=True, proj='pcarree')

# formatação dos eixos
ax.format(coast=False, borders=False, innerborders=False,
          labels=True, latlines=2, lonlines=3,
          latlim=(latmin, latmax), lonlim=(lonmin, lonmax),
          small='40px', large='45px',
          suptitle='GOES CH13 (10.35 µm)')

# carrega tabela de cores
cpt_ir = loadCPT(f'/content/ir.cpt')
cmap_ir = cm.colors.LinearSegmentedColormap('cpt_ir', cpt_ir)

#========================================================================================================================#
#                                              LOOP DAS IMAGENS
#========================================================================================================================#
# Loop das imagens
for i, date_image in enumerate(pd.date_range('202601242200', '202601242350', freq='10min')):

    #--------------------------------------------------------------------------#
    #                          DATA E HORÁRIO
    #--------------------------------------------------------------------------#
    # data
    yyyymmddhhmn = date_image.strftime('%Y%m%d%H%M') # 201904202240

    # define o satélite: GOES-16 ou GOES-19
    start_g19 = datetime(2025,4,7,0,0)
    imagem_atual = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M')
    goes_number = '16' if imagem_atual < start_g19 else '19'

    # extrai o ano, mês, dia, hor e min
    yyyy = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%Y')
    mm = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%m')
    dd = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%d')
    hh = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%H')
    mn = datetime.strptime(yyyymmddhhmn, '%Y%m%d%H%M').strftime('%M')

    print('#=====================================================================================================#')
    print(f'                           PROCESSANDO A IMAGEM = {yyyy}-{mm}-{dd} {hh}{mn} UTC'                       )
    print('#=====================================================================================================#')

    # download do arquivo
    file_name = download_CMI(yyyymmddhhmn, band, goes_number, input)

    # caminho do arquivo que foi baixado
    path = f'{input}/{file_name}.nc'

    #--------------------------------------------------------------------------#
    #                    REPROJETA OS DADOS DO ABI
    #--------------------------------------------------------------------------#
    # chama a função que faz a reprojeção (file, variable, extent, resolution)
    grid = remap(path, 'CMI', extent, 2)

    # remove os arquivos desnecessários
    !rm {input}/{file_name}.nc
    !rm {input}/{file_name}.nc.aux.xml

    #--------------------------------------------------------------------------#
    #                         Leitura do dado
    #--------------------------------------------------------------------------#
    # leitura do arquivo reprojetado
    dataset_ir = xr.open_dataset(f'{input}/{file_name}_ret.nc', mask_and_scale=True).sel(lon=slice(lonmin, lonmax), lat=slice(latmax, latmin))

    # aplica o fator de escala (/10) e transforma para Graus Celsius
    dataset_ir['Band1'] = (dataset_ir['Band1']/10.) - 273.15

    #--------------------------------------------------------------------------#
    #                        Plotando a figura
    #--------------------------------------------------------------------------#
    if i == 0:
        map1 = ax[i].imshow(dataset_ir['Band1'],
                            cmap=cmap_ir,
                            extent=[lonmin, lonmax, latmin, latmax],
                            levels=uplt.arange(-103.0, 105, 1.0))
    else:
        ax[i].imshow(dataset_ir['Band1'],
                     cmap=cmap_ir,
                     extent=[lonmin, lonmax, latmin, latmax],
                     levels=uplt.arange(-103.0, 105, 1.0))

    # plota contornos dos Estados
    shapefile = list(shpreader.Reader('https://github.com/evmpython/Minicurso_UFCG_nov_2025/raw/main/utils/BR_UF_2019.shp').geometries())
    ax[i].add_geometries(shapefile, ccrs.PlateCarree(), edgecolor='yellow', facecolor='none', linewidth=2.1)

    # plota titulo de cada figura
    if (i==1 or i==2 or i==4 or i==5 or i==7 or i==8): ax[i].format(title=f'{yyyy}-{mm}-{dd} {hh}:{mn} UTC', labels=False, linewidth=3)
    if (i==10 or i==11): ax[i].format(title=f'{yyyy}-{mm}-{dd} {hh}:{mn} UTC', labels=[False, False, True, False], linewidth=3)
    if (i==0 or i==3 or i==6): ax[i].format(title=f'{yyyy}-{mm}-{dd} {hh}:{mn} UTC', labels=[True, False, False, False], linewidth=3)
    if (i==9): ax[i].format(title=f'{yyyy}-{mm}-{dd} {hh}:{mn} UTC', labels=[True, False, True, False], linewidth=3)
    print('\n')

# plota barra de cores
fig.colorbar(map1, loc='b', label='Temperatura de Brilho ($\degree$C)', ticks=25, ticklabelsize=40, labelsize=40, width=0.5, length=0.7)

# salva figura
fig.savefig(f'{output}/SAT_05_G{goes_number}_ch{band}_painel_retangular_trealcada.jpg', dpi=300, bbox_inches='tight')

Output hidden; open in https://colab.research.google.com to view.